# Silver validation with Great Expectations (streaming)

Purpose: validate the Silver cleaned stream incrementally, emit append-only observability outputs, and publish append-only validated/quarantine sinks for downstream consumers.


In [0]:
%pip install great-expectations==0.18.21


  Using cached great_expectations-0.18.21-py3-none-any.whl.metadata (8.5 kB)
  Using cached altair-4.2.2-py3-none-any.whl.metadata (13 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached makefun-1.16.0-py2.py3-none-any.whl.metadata (2.9 kB)
  Using cached ruamel.yaml-0.17.40-py3-none-any.whl.metadata (19 kB)
  Using cached tzlocal-5.3.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached entrypoints-0.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached toolz-1.1.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached ruamel_yaml_clib-0.2.15-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (3.5 kB)
Using cached great_expectations-0.18.21-py3-none-any.whl (5.4 MB)
Using cached altair-4.2.2-py3-none-any.whl (813 kB)
Using cached colorama-0.4.6-py2.py3-none-any.whl (25 kB)
Using cached makefun-1.16.0-py2.py3-none-any.w

In [0]:
import json
import os
import traceback
from datetime import datetime, timezone
from functools import reduce

import great_expectations as gx
from great_expectations.data_context import FileDataContext
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "hsl"

STORAGE_ACCOUNT = "streanmingdatasta"
LAKEHOUSE_CONTAINER = "lakehouse"
BASE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/external/hant-catalog/silver"

SILVER_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.silver_vehicle_positions_cleaned"
SILVER_VALIDATED_OUTPUT_PATH = f"{BASE_PATH}/silver_validated_stream/hsl_vehicle_positions"
SILVER_QUARANTINE_OUTPUT_PATH = f"{BASE_PATH}/silver_ge_quarantine_stream/hsl_vehicle_positions"
SILVER_GE_RESULTS_PATH = f"{BASE_PATH}/silver_ge/ge_results/hsl_vehicle_positions"
SILVER_GE_DETAILS_PATH = f"{BASE_PATH}/silver_ge/ge_details/hsl_vehicle_positions"
SILVER_FAILED_SAMPLES_PATH = f"{BASE_PATH}/silver_ge/failed_row_samples/hsl_vehicle_positions"
SILVER_RULE_METRICS_PATH = f"{BASE_PATH}/silver_ge/rule_metrics/hsl_vehicle_positions"
SILVER_ALERTS_PATH = f"{BASE_PATH}/silver_ge/alerts/hsl_vehicle_positions"
SILVER_GATE_RESULTS_PATH = f"{BASE_PATH}/silver_ge/gate_results/hsl_vehicle_positions"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/silver_ge_validate_stream_hsl_vehicle_positions"

SILVER_VALIDATED_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.silver_vehicle_positions_validated_stream"
SILVER_VALIDATION_QUAR_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.silver_vehicle_positions_ge_quarantine_stream"
SILVER_GE_RESULTS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.silver_ge_results_hsl_vehicle_positions"
SILVER_GE_DETAILS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.silver_ge_details_hsl_vehicle_positions"
SILVER_GE_SAMPLES_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.silver_ge_failed_samples_hsl_vehicle_positions"
SILVER_RULE_METRICS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.silver_ge_rule_metrics_hsl_vehicle_positions"
SILVER_ALERTS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.silver_ge_alerts_hsl_vehicle_positions"
SILVER_GATE_RESULTS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.silver_gate_results_hsl_vehicle_positions"

TRIGGER_INTERVAL = "10 seconds"
MAX_FAILED_SAMPLE_ROWS = 500
MAX_CRITICAL_ROWS = 0
MAX_HIGH_ROWS = 0
MAX_QUARANTINE_RATE = 0.02
MAX_MEDIUM_WARNING_RATE = 0.05

CONTEXT_ROOT_DIR = "/dbfs/great_expectations/silver_validate_ge_stream_ctx"
DATASOURCE_NAME = "silver_runtime_spark_stream"
DATA_ASSET_NAME = "silver_vehicle_positions_cleaned_stream_asset"
EXPECTATION_SUITE_NAME = "silver_clean_validation_stream_suite"
QUERY_NAME = "silver_vehicle_positions_ge_validate_stream"


def filter_to_target_ingest_date(df):
    if "ingest_date" not in df.columns:
        raise ValueError("Expected ingest_date column before validation date filtering.")
    return df.where(F.to_date(F.col("ingest_date")) == F.current_date())


In [0]:
# tables_to_drop = [
#     SILVER_VALIDATED_TABLE,
#     SILVER_VALIDATION_QUAR_TABLE,
#     SILVER_GE_RESULTS_TABLE,
#     SILVER_GE_DETAILS_TABLE,
#     SILVER_GE_SAMPLES_TABLE,
#     SILVER_RULE_METRICS_TABLE,
#     SILVER_ALERTS_TABLE,
#     SILVER_GATE_RESULTS_TABLE,
# ]

# for table in tables_to_drop:
#     spark.sql(f"DROP TABLE IF EXISTS {table}")


# dbutils.fs.rm(CHECKPOINT_PATH, True)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

for q in spark.streams.active:
    if q.name == QUERY_NAME:
        q.stop()

base_schema = spark.table(SILVER_TABLE).schema
sink_schema = StructType(list(base_schema.fields) + [
    StructField("silver_failed_rule_ids", ArrayType(StringType()), True),
    StructField("silver_failed_severities", ArrayType(StringType()), True),
    StructField("silver_failed_rule_descriptions", ArrayType(StringType()), True),
    StructField("silver_failed_rule_count", LongType(), True),
    StructField("silver_validation_run_id", StringType(), True),
    StructField("silver_validation_batch_id", LongType(), True),
    StructField("silver_validation_scope", StringType(), True),
    StructField("silver_validation_ts", TimestampType(), True),
    StructField("silver_validation_date", StringType(), True),
    StructField("silver_validation_status", StringType(), True),
])

def precreate_sink(path: str, table_name: str):
    if not DeltaTable.isDeltaTable(spark, path):
        (
            spark.createDataFrame([], sink_schema)
            .write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .partitionBy("ingest_date")
            .save(path)
        )
    spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {table_name}
    USING DELTA
    LOCATION "{path}"
    ''')

precreate_sink(SILVER_VALIDATED_OUTPUT_PATH, SILVER_VALIDATED_TABLE)
precreate_sink(SILVER_QUARANTINE_OUTPUT_PATH, SILVER_VALIDATION_QUAR_TABLE)


In [0]:
gate_schema = StructType([
    StructField("run_id", StringType()),
    StructField("validated_table", StringType()),
    StructField("run_ts_utc", TimestampType()),
    StructField("validation_date", StringType()),
    StructField("validation_scope", StringType()),
    StructField("input_row_count", LongType()),
    StructField("validated_row_count", LongType()),
    StructField("quarantined_row_count", LongType()),
    StructField("quarantine_rate", DoubleType()),
    StructField("critical_failed_rows", LongType()),
    StructField("high_failed_rows", LongType()),
    StructField("medium_failed_rows", LongType()),
    StructField("low_failed_rows", LongType()),
    StructField("warning_rate", DoubleType()),
    StructField("gate_status", StringType()),
    StructField("gate_reason", StringType()),
    StructField("gold_allowed", BooleanType()),
    StructField("validated_output_path", StringType()),
    StructField("quarantine_output_path", StringType()),
])


def register_table(path: str, table_name: str):
    spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {table_name}
    USING DELTA
    LOCATION "{path}"
    ''')


def write_append(df, path: str, table_name: str, merge_schema: bool = True, partition_cols: list[str] | None = None):
    if not DeltaTable.isDeltaTable(spark, path):
        writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        if merge_schema:
            writer = writer.option("mergeSchema", "true")
        if partition_cols:
            writer = writer.partitionBy(*partition_cols)
        writer.save(path)
    else:
        writer = df.write.format("delta").mode("append")
        if merge_schema:
            writer = writer.option("mergeSchema", "true")
        if partition_cols:
            writer = writer.partitionBy(*partition_cols)
        writer.save(path)
    register_table(path, table_name)


def write_gate_results(rows):
    write_append(spark.createDataFrame(rows, schema=gate_schema), SILVER_GATE_RESULTS_PATH, SILVER_GATE_RESULTS_TABLE, merge_schema=False)


def union_by_name_allow_missing(dfs):
    if not dfs:
        return None
    return reduce(lambda left, right: left.unionByName(right, allowMissingColumns=True), dfs)


def gx_to_dict(obj):
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "to_json_dict"):
        return obj.to_json_dict()
    if hasattr(obj, "to_dict"):
        return obj.to_dict()
    return {}


In [0]:
def get_gx_context():
    os.makedirs(CONTEXT_ROOT_DIR, exist_ok=True)
    print(json.dumps({
        "layer": "silver_validate_ge",
        "message": "Initializing Great Expectations context",
        "context_root_dir": CONTEXT_ROOT_DIR,
    }, default=str))
    context = FileDataContext.create(project_root_dir=CONTEXT_ROOT_DIR)
    return context


def apply_silver_expectations(validator):
    for column_name in [
        "event_type", "transport_mode", "topic_operator_id", "topic_vehicle_number", "topic_direction_id", "topic_start_time",
        "topic_next_stop_id", "dir", "operator_id", "vehicle_number", "event_ts_raw", "event_ts", "event_ts_unix", "latitude",
        "longitude", "acceleration", "delay_sec", "door_status", "location_source", "stop_id", "occupancy", "route_id",
        "vehicle_id", "business_key", "direction_id", "operating_day", "silver_ingest_ts", "topic_vehicle_number_norm",
        "topic_operator_id_norm", "journey_start_time", "topic_route_id", "payload_route_id", "speed", "heading",
        "silver_event_date", "event_to_silver_delay_sec",
    ]:
        validator.expect_column_to_exist(column_name)

    validator.expect_column_values_to_not_be_null("event_ts")
    validator.expect_column_values_to_not_be_null("event_ts_unix")
    validator.expect_column_values_to_not_be_null("latitude")
    validator.expect_column_values_to_not_be_null("longitude")
    validator.expect_column_values_to_not_be_null("route_id")
    validator.expect_column_values_to_not_be_null("vehicle_id")
    validator.expect_column_values_to_not_be_null("business_key")
    validator.expect_column_values_to_not_be_null("silver_ingest_ts")
    validator.expect_column_values_to_not_be_null("silver_event_date")
    validator.expect_column_values_to_not_be_null("operating_day")
    validator.expect_column_values_to_not_be_null("event_type")
    validator.expect_column_values_to_not_be_null("transport_mode")
    validator.expect_column_values_to_not_be_null("topic_operator_id")
    validator.expect_column_values_to_not_be_null("topic_vehicle_number")
    validator.expect_column_values_to_not_be_null("topic_direction_id")
    validator.expect_column_values_to_not_be_null("topic_start_time")
    validator.expect_column_values_to_not_be_null("dir")
    validator.expect_column_values_to_not_be_null("operator_id")
    validator.expect_column_values_to_not_be_null("vehicle_number")
    validator.expect_column_values_to_not_be_null("event_ts_raw")
    validator.expect_column_values_to_not_be_null("journey_start_time")

    validator.expect_column_values_to_be_in_set("event_type", ["vp"])
    validator.expect_column_values_to_be_in_set("transport_mode", ["bus", "tram", "train", "metro", "ferry", "ubus", "robot"])
    validator.expect_column_values_to_be_in_set("topic_direction_id", ["1", "2"])
    validator.expect_column_values_to_be_in_set("dir", ["1", "2"])
    validator.expect_column_values_to_be_in_set("direction_id", ["1", "2"])
    validator.expect_column_values_to_be_in_set("door_status", [0, 1])
    validator.expect_column_values_to_be_in_set("location_source", ["GPS", "ODO", "MAN", "DR", "N/A"])

    validator.expect_column_values_to_match_regex("topic_operator_id", r"^\d{4}$") # topic_operator_id should be 4 digits
    validator.expect_column_values_to_match_regex("topic_vehicle_number", r"^\d{5}$") # topic_vehicle_number should be 5 digits
    validator.expect_column_values_to_match_regex("topic_start_time", r"^(?:[01]\d|2[0-3]):[0-5]\d$") # topic_start_time should be in HH:mm format
    validator.expect_column_values_to_match_regex("journey_start_time", r"^(?:[01]\d|2[0-3]):[0-5]\d$") # journey_start_time should be in HH:mm format
    validator.expect_column_values_to_match_regex("event_ts_raw", r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d{1,6})?Z$") # event_ts_raw should be in ISO-8601 UTC format
    validator.expect_column_values_to_match_regex("topic_next_stop_id", r"^(\d+|EOL)$", mostly=0.99) # topic_next_stop_id should be either digits or "EOL"

    validator.expect_column_values_to_be_between("latitude", min_value=59.0, max_value=61.5) # latitude should be within Helsinki region
    validator.expect_column_values_to_be_between("longitude", min_value=23.0, max_value=26.5) # longitude should be within Helsinki region
    validator.expect_column_values_to_be_between("speed", min_value=0, max_value=50) # speed should be between 0 and 50 m/s (180 km/h)
    validator.expect_column_values_to_be_between("heading", min_value=0, max_value=360) # heading should be between 0 and 360 degrees
    validator.expect_column_values_to_be_between("occupancy", min_value=0, max_value=100) # occupancy should be between 0 and 100 percent
    validator.expect_column_values_to_be_unique("business_key")

    validator.expect_column_pair_values_to_be_equal("topic_vehicle_number_norm", "vehicle_number")
    validator.expect_column_pair_values_to_be_equal("topic_operator_id_norm", "operator_id")
    validator.expect_column_pair_values_to_be_equal("topic_start_time", "journey_start_time")
    validator.expect_column_pair_values_to_be_equal("topic_route_id", "payload_route_id")


def run_silver_ge_validation(batch_df, batch_run_id: str, batch_run_ts):
    context = get_gx_context()
    context.add_or_update_expectation_suite(expectation_suite_name=EXPECTATION_SUITE_NAME)
    datasource = context.sources.add_or_update_spark(name=DATASOURCE_NAME)
    asset = datasource.add_dataframe_asset(name=f"{DATA_ASSET_NAME}_{batch_run_id}")
    batch_request = asset.build_batch_request(dataframe=batch_df)
    validator = context.get_validator(batch_request=batch_request, expectation_suite_name=EXPECTATION_SUITE_NAME)
    apply_silver_expectations(validator)
    validator.save_expectation_suite(discard_failed_expectations=False)
    validation_result = validator.validate(
        run_id={"run_name": batch_run_id, "run_time": batch_run_ts.isoformat()},
        data_context=context,
    )
    return gx_to_dict(validation_result)


In [0]:
rule_specs = [
    {"rule_id": "critical_event_ts_null", "severity": "critical", "description": "Event timestamp must be present in Silver.", "fail_condition": F.col("event_ts").isNull(), "quarantine_row": True},
    {"rule_id": "critical_event_ts_unix_null", "severity": "critical", "description": "Event unix timestamp must be present in Silver.", "fail_condition": F.col("event_ts_unix").isNull(), "quarantine_row": True},
    {"rule_id": "critical_vehicle_id_null", "severity": "critical", "description": "Vehicle identifier must be present in Silver.", "fail_condition": F.col("vehicle_id").isNull() | (F.trim(F.col("vehicle_id")) == ""), "quarantine_row": True},
    {"rule_id": "high_route_id_null", "severity": "high", "description": "Route identifier must be present in Silver.", "fail_condition": F.col("route_id").isNull() | (F.trim(F.col("route_id")) == ""), "quarantine_row": True},
    {"rule_id": "high_latitude_out_of_bounds", "severity": "high", "description": "Latitude must fall within HSL operating bounds.", "fail_condition": F.col("latitude").isNull() | (F.col("latitude") < F.lit(59.0)) | (F.col("latitude") > F.lit(61.5)), "quarantine_row": True},
    {"rule_id": "high_longitude_out_of_bounds", "severity": "high", "description": "Longitude must fall within HSL operating bounds.", "fail_condition": F.col("longitude").isNull() | (F.col("longitude") < F.lit(23.0)) | (F.col("longitude") > F.lit(26.5)), "quarantine_row": True},
    {"rule_id": "high_event_type_unexpected", "severity": "high", "description": "Silver vehicle-position stream is expected to contain only VP events.", "fail_condition": F.col("event_type").isNotNull() & (F.col("event_type") != F.lit("vp")), "quarantine_row": True},
    {"rule_id": "medium_topic_operator_id_invalid_format", "severity": "medium", "description": "Topic operator ID should be exactly 4 digits.", "fail_condition": F.col("topic_operator_id").isNotNull() & ~F.col("topic_operator_id").rlike(r"^\d{4}$"), "quarantine_row": True},
    {"rule_id": "medium_topic_vehicle_number_invalid_format", "severity": "medium", "description": "Topic vehicle number should be exactly 5 digits.", "fail_condition": F.col("topic_vehicle_number").isNotNull() & ~F.col("topic_vehicle_number").rlike(r"^\d{5}$"), "quarantine_row": True},
    {"rule_id": "medium_topic_direction_invalid", "severity": "medium", "description": "Topic direction should be 1 or 2.", "fail_condition": F.col("topic_direction_id").isNotNull() & ~F.col("topic_direction_id").isin("1", "2"), "quarantine_row": True},
    {"rule_id": "medium_payload_dir_invalid", "severity": "medium", "description": "Payload dir should be 1 or 2.", "fail_condition": F.col("dir").isNotNull() & ~F.col("dir").isin("1", "2"), "quarantine_row": True},
    {"rule_id": "medium_transport_mode_unexpected", "severity": "medium", "description": "Transport mode should be in the expected domain.", "fail_condition": F.col("transport_mode").isNotNull() & ~F.col("transport_mode").isin("bus", "tram", "train", "metro", "ferry", "ubus", "robot"), "quarantine_row": True},
    {"rule_id": "medium_negative_speed", "severity": "medium", "description": "Speed should not be negative.", "fail_condition": F.col("speed").isNotNull() & (F.col("speed") < F.lit(0.0)), "quarantine_row": True},
    {"rule_id": "medium_heading_out_of_range", "severity": "medium", "description": "Heading should be between 0 and 360.", "fail_condition": F.col("heading").isNotNull() & ((F.col("heading") < F.lit(0)) | (F.col("heading") > F.lit(360))), "quarantine_row": True},
    {"rule_id": "medium_occupancy_out_of_range", "severity": "medium", "description": "Occupancy should be between 0 and 100 when present.", "fail_condition": F.col("occupancy").isNotNull() & ((F.col("occupancy") < F.lit(0)) | (F.col("occupancy") > F.lit(100))), "quarantine_row": True},
    {"rule_id": "medium_door_status_invalid", "severity": "medium", "description": "Door status should be 0 or 1 when present.", "fail_condition": F.col("door_status").isNotNull() & ~F.col("door_status").isin(0, 1), "quarantine_row": True},
    {"rule_id": "medium_event_ts_raw_invalid_format", "severity": "medium", "description": "Raw event timestamp should be ISO 8601 UTC format.", "fail_condition": F.col("event_ts_raw").isNotNull() & ~F.col("event_ts_raw").rlike(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:\.\d{1,6})?Z$"), "quarantine_row": True},
    {"rule_id": "medium_event_ts_vs_unix_mismatch", "severity": "medium", "description": "Parsed event timestamp and unix timestamp should agree.", "fail_condition": F.col("event_ts").isNotNull() & F.col("event_ts_unix").isNotNull() & (F.abs(F.col("event_ts").cast("long") - F.col("event_ts_unix")) > F.lit(1)), "quarantine_row": True},
    {"rule_id": "medium_event_future_gt_2m", "severity": "medium", "description": "Event timestamp should not be more than 2 minutes ahead of Silver ingest time.", "fail_condition": F.col("event_to_silver_delay_sec").isNotNull() & (F.col("event_to_silver_delay_sec") < F.lit(-120)), "quarantine_row": False},
    {"rule_id": "medium_event_stale_gt_5m", "severity": "medium", "description": "Event timestamp is more than 5 minutes older than Silver ingest time.", "fail_condition": F.col("event_to_silver_delay_sec").isNotNull() & (F.col("event_to_silver_delay_sec") > F.lit(300)), "quarantine_row": False},
    {"rule_id": "low_location_source_unexpected", "severity": "low", "description": "Location source should be in the expected domain.", "fail_condition": F.col("location_source").isNotNull() & ~F.col("location_source").isin("GPS", "ODO", "MAN", "DR", "N/A"), "quarantine_row": True},
    {"rule_id": "low_topic_start_time_invalid_format", "severity": "low", "description": "Topic start time should follow HH:mm in 24-hour format.", "fail_condition": F.col("topic_start_time").isNotNull() & ~F.col("topic_start_time").rlike(r"^(?:[01]\d|2[0-3]):[0-5]\d$"), "quarantine_row": True},
    {"rule_id": "low_journey_start_time_invalid_format", "severity": "low", "description": "Payload journey start time should follow HH:mm in 24-hour format.", "fail_condition": F.col("journey_start_time").isNotNull() & ~F.col("journey_start_time").rlike(r"^(?:[01]\d|2[0-3]):[0-5]\d$"), "quarantine_row": True},
    {"rule_id": "low_topic_route_mismatch", "severity": "low", "description": "Topic route and payload route should usually agree.", "fail_condition": F.col("topic_route_id").isNotNull() & F.col("payload_route_id").isNotNull() & (F.col("topic_route_id") != F.col("payload_route_id")), "quarantine_row": True},
    {"rule_id": "low_topic_vehicle_mismatch", "severity": "low", "description": "Topic vehicle number and payload vehicle number should usually agree after normalization.", "fail_condition": F.col("topic_vehicle_number_norm").isNotNull() & F.col("vehicle_number").isNotNull() & (F.col("topic_vehicle_number_norm") != F.col("vehicle_number")), "quarantine_row": True},
    {"rule_id": "low_topic_operator_mismatch", "severity": "low", "description": "Topic operator ID and payload operator ID should usually agree after normalization.", "fail_condition": F.col("topic_operator_id_norm").isNotNull() & F.col("operator_id").isNotNull() & (F.col("topic_operator_id_norm") != F.col("operator_id")), "quarantine_row": True},
    {"rule_id": "low_topic_direction_mismatch", "severity": "low", "description": "Topic direction and canonical direction should usually agree.", "fail_condition": F.col("topic_direction_id").isNotNull() & F.col("direction_id").isNotNull() & (F.col("topic_direction_id") != F.col("direction_id")), "quarantine_row": True},
    {"rule_id": "low_topic_start_time_mismatch", "severity": "low", "description": "Topic start time and payload journey start time should usually agree.", "fail_condition": F.col("topic_start_time").isNotNull() & F.col("journey_start_time").isNotNull() & (F.col("topic_start_time") != F.col("journey_start_time")), "quarantine_row": True},
    {"rule_id": "low_operating_day_vs_event_date_mismatch", "severity": "low", "description": "Operating day should usually match the event date.", "fail_condition": F.col("operating_day").isNotNull() & F.col("event_ts").isNotNull() & (F.col("operating_day") != F.to_date(F.col("event_ts"))), "quarantine_row": False},
]


In [0]:
def write_silver_validation_batch(batch_df, batch_id: int):
    if batch_df.isEmpty():
        return

    batch_df = filter_to_target_ingest_date(batch_df)
    if batch_df.isEmpty():
        return

    batch_run_ts = datetime.now(timezone.utc)
    batch_run_id = f"silver_ge_stream_{batch_id}_{batch_run_ts.strftime('%Y%m%d_%H%M%S')}"
    validation_date = batch_run_ts.date().isoformat()
    validation_scope = f"microbatch_id={batch_id}"

    working_df = batch_df.cache()

    try:
        row_count = working_df.count()
        print(json.dumps({
            "layer": "silver_validate_ge",
            "message": "Starting foreachBatch validation",
            "batch_id": int(batch_id),
            "batch_run_id": batch_run_id,
            "row_count": int(row_count),
            "validation_scope": validation_scope,
            "context_root_dir": CONTEXT_ROOT_DIR,
        }, default=str))

        validation_result = run_silver_ge_validation(working_df, batch_run_id, batch_run_ts)
        print(json.dumps({
            "layer": "silver_validate_ge",
            "batch_run_id": batch_run_id,
        }, default=str))
        stats = validation_result.get("statistics", {}) or {}

        summary_row = [{
            "run_id": batch_run_id,
            "validated_table": SILVER_TABLE,
            "validation_scope": validation_scope,
            "validation_date": validation_date,
            "run_ts_utc": batch_run_ts,
            "row_count": int(row_count),
            "success": bool(validation_result.get("success", False)),
            "evaluated_expectations": int(stats.get("evaluated_expectations", 0) or 0),
            "successful_expectations": int(stats.get("successful_expectations", 0) or 0),
            "unsuccessful_expectations": int(stats.get("unsuccessful_expectations", 0) or 0),
            "success_percent": float(stats.get("success_percent", 0.0) or 0.0),
        }]
        write_append(spark.createDataFrame(summary_row),
                     SILVER_GE_RESULTS_PATH, SILVER_GE_RESULTS_TABLE)

        detail_rows = []
        for result in validation_result.get("results", []):
            cfg = result.get("expectation_config", {}) or {}
            detail_rows.append({
                "run_id": batch_run_id,
                "validated_table": SILVER_TABLE,
                "validation_date": validation_date,
                "run_ts_utc": batch_run_ts,
                "expectation_type": cfg.get("expectation_type"),
                "column_name": (cfg.get("kwargs", {}) or {}).get("column"),
                "success": bool(result.get("success", False)),
                "result_json": json.dumps(result.get("result", {}), default=str),
                "kwargs_json": json.dumps((cfg.get("kwargs", {}) or {}), default=str),
            })
        if detail_rows:
            write_append(spark.createDataFrame(detail_rows),
                         SILVER_GE_DETAILS_PATH, SILVER_GE_DETAILS_TABLE)

        failed_dfs = []
        for spec in rule_specs:
            failed_dfs.append(
                working_df
                .filter(spec["fail_condition"])
                .withColumn("rule_id", F.lit(spec["rule_id"]))
                .withColumn("severity", F.lit(spec["severity"]))
                .withColumn("rule_description", F.lit(spec["description"]))
                .withColumn("quarantine_row", F.lit(spec["quarantine_row"]))
                .withColumn("run_id", F.lit(batch_run_id))
                .withColumn("validation_date", F.lit(validation_date))
                .withColumn("validation_scope", F.lit(validation_scope))
                .withColumn("silver_validation_run_id", F.lit(batch_run_id))
                .withColumn("silver_validation_batch_id", F.lit(int(batch_id)).cast("bigint"))
                .withColumn("silver_validation_scope", F.lit(validation_scope))
                .withColumn("silver_validation_ts", F.lit(batch_run_ts))
                .withColumn("silver_validation_date", F.lit(validation_date))
            )

        duplicate_keys_df = working_df.groupBy("business_key").count().filter(
            F.col("count") > 1).select("business_key")
        if duplicate_keys_df.count() > 0:
            failed_dfs.append(
                working_df.alias("s")
                .join(duplicate_keys_df.alias("d"), on=["business_key"], how="inner")
                .withColumn("rule_id", F.lit("critical_duplicate_business_key"))
                .withColumn("severity", F.lit("critical"))
                .withColumn("rule_description", F.lit("Business key must be unique within the Silver validation microbatch."))
                .withColumn("quarantine_row", F.lit(True))
                .withColumn("run_id", F.lit(batch_run_id))
                .withColumn("validation_date", F.lit(validation_date))
                .withColumn("validation_scope", F.lit(validation_scope))
                .withColumn("silver_validation_run_id", F.lit(batch_run_id))
                .withColumn("silver_validation_batch_id", F.lit(int(batch_id)).cast("bigint"))
                .withColumn("silver_validation_scope", F.lit(validation_scope))
                .withColumn("silver_validation_ts", F.lit(batch_run_ts))
                .withColumn("silver_validation_date", F.lit(validation_date))
            )

        all_failed_rows_df = union_by_name_allow_missing(failed_dfs)
        if all_failed_rows_df is None:
            all_failed_rows_df = spark.createDataFrame(
                [],
                schema=(
                    working_df
                    .withColumn("rule_id", F.lit(None).cast("string"))
                    .withColumn("severity", F.lit(None).cast("string"))
                    .withColumn("rule_description", F.lit(None).cast("string"))
                    .withColumn("quarantine_row", F.lit(None).cast("boolean"))
                    .withColumn("run_id", F.lit(None).cast("string"))
                    .withColumn("validation_date", F.lit(None).cast("string"))
                    .withColumn("validation_scope", F.lit(None).cast("string"))
                    .withColumn("silver_validation_run_id", F.lit(None).cast("string"))
                    .withColumn("silver_validation_batch_id", F.lit(None).cast("bigint"))
                    .withColumn("silver_validation_scope", F.lit(None).cast("string"))
                    .withColumn("silver_validation_ts", F.lit(None).cast("timestamp"))
                    .withColumn("silver_validation_date", F.lit(None).cast("string"))
                    .schema
                ),
            )

        failed_rollup_df = (
            all_failed_rows_df
            .filter(F.col("rule_id").isNotNull())
            .groupBy("business_key")
            .agg(
                F.collect_set("rule_id").alias("silver_failed_rule_ids"),
                F.collect_set("severity").alias("silver_failed_severities"),
                F.collect_set("rule_description").alias(
                    "silver_failed_rule_descriptions"),
                F.max(F.when(F.col("quarantine_row") == True, F.lit(1)
                             ).otherwise(F.lit(0))).alias("should_quarantine"),
            )
        )

        empty_array = F.expr("array()").cast("array<string>")
        classified_df = (
            working_df
            .join(failed_rollup_df, on="business_key", how="left")
            .withColumn("silver_validation_run_id", F.lit(batch_run_id))
            .withColumn("silver_validation_batch_id", F.lit(int(batch_id)).cast("bigint"))
            .withColumn("silver_validation_scope", F.lit(validation_scope))
            .withColumn("silver_validation_ts", F.lit(batch_run_ts))
            .withColumn("silver_validation_date", F.lit(validation_date))
            .withColumn("silver_failed_rule_ids", F.when(F.col("silver_failed_rule_ids").isNull(), empty_array).otherwise(F.col("silver_failed_rule_ids")))
            .withColumn("silver_failed_severities", F.when(F.col("silver_failed_severities").isNull(), empty_array).otherwise(F.col("silver_failed_severities")))
            .withColumn("silver_failed_rule_descriptions", F.when(F.col("silver_failed_rule_descriptions").isNull(), empty_array).otherwise(F.col("silver_failed_rule_descriptions")))
            .withColumn("silver_failed_rule_count", F.size(F.col("silver_failed_rule_ids")).cast("bigint"))
            .withColumn(
                "silver_validation_status",
                F.when(F.coalesce(F.col("should_quarantine"), F.lit(0))
                       == F.lit(1), F.lit("quarantined"))
                .otherwise(F.lit("validated"))
            )
            .drop("should_quarantine")
        )

        validated_df = classified_df.filter(
            F.col("silver_validation_status") == "validated").cache()
        quarantine_df = classified_df.filter(
            F.col("silver_validation_status") == "quarantined").cache()
        try:
            validated_row_count = validated_df.count()
            quarantined_row_count = quarantine_df.count()
            quarantine_rate = float(quarantined_row_count) / \
                float(row_count) if row_count else 0.0
            write_append(validated_df, SILVER_VALIDATED_OUTPUT_PATH,
                         SILVER_VALIDATED_TABLE, merge_schema=False, partition_cols=["ingest_date"])
            write_append(quarantine_df, SILVER_QUARANTINE_OUTPUT_PATH,
                         SILVER_VALIDATION_QUAR_TABLE, merge_schema=False, partition_cols=["ingest_date"])
        finally:
            validated_df.unpersist()
            quarantine_df.unpersist()

        rule_metrics_df = (
            all_failed_rows_df
            .filter(F.col("rule_id").isNotNull())
            .groupBy("run_id", "validation_date", "validation_scope", "rule_id", "severity", "rule_description")
            .agg(F.count("*").alias("failed_rule_rows"), F.countDistinct("business_key").alias("affected_rows"))
        )
        if not rule_metrics_df.isEmpty():
            write_append(rule_metrics_df, SILVER_RULE_METRICS_PATH,
                         SILVER_RULE_METRICS_TABLE)

        failed_samples_df = all_failed_rows_df.filter(
            F.col("rule_id").isNotNull()).limit(MAX_FAILED_SAMPLE_ROWS)
        if not failed_samples_df.isEmpty():
            write_append(failed_samples_df,
                         SILVER_FAILED_SAMPLES_PATH, SILVER_GE_SAMPLES_TABLE)

        alert_rows = []
        for row in rule_metrics_df.collect():
            if row["severity"] in ("critical", "high"):
                alert_rows.append({
                    "run_id": row["run_id"],
                    "alert_ts": batch_run_ts,
                    "severity": row["severity"],
                    "rule_id": row["rule_id"],
                    "message": f"{row['rule_id']} affected {row['affected_rows']} Silver rows",
                    "failed_rows": int(row["affected_rows"]),
                })
        if alert_rows:
            write_append(spark.createDataFrame(alert_rows),
                         SILVER_ALERTS_PATH, SILVER_ALERTS_TABLE)

        severity_rollup = {
            row["severity"]: row["affected_rows"]
            for row in (
                all_failed_rows_df
                .filter(F.col("rule_id").isNotNull())
                .select("severity", "business_key")
                .distinct()
                .groupBy("severity")
                .agg(F.count("*").alias("affected_rows"))
                .collect()
            )
        }

        critical_failed_rows = int(severity_rollup.get("critical", 0))
        high_failed_rows = int(severity_rollup.get("high", 0))
        medium_failed_rows = int(severity_rollup.get("medium", 0))
        low_failed_rows = int(severity_rollup.get("low", 0))
        warning_rows = medium_failed_rows + low_failed_rows
        warning_rate = float(warning_rows) / \
            float(row_count) if row_count else 0.0

        block_reasons = []
        warn_reasons = []
        if critical_failed_rows > MAX_CRITICAL_ROWS:
            block_reasons.append(
                f"critical_failed_rows={critical_failed_rows} exceeds threshold {MAX_CRITICAL_ROWS}")
        if high_failed_rows > MAX_HIGH_ROWS:
            block_reasons.append(
                f"high_failed_rows={high_failed_rows} exceeds threshold {MAX_HIGH_ROWS}")
        if quarantine_rate > MAX_QUARANTINE_RATE:
            block_reasons.append(
                f"quarantine_rate={quarantine_rate:.6f} exceeds threshold {MAX_QUARANTINE_RATE:.6f}")
        if medium_failed_rows > 0:
            warn_reasons.append(f"medium_failed_rows={medium_failed_rows}")
        if low_failed_rows > 0:
            warn_reasons.append(f"low_failed_rows={low_failed_rows}")
        if warning_rate > MAX_MEDIUM_WARNING_RATE:
            warn_reasons.append(
                f"warning_rate={warning_rate:.6f} exceeds threshold {MAX_MEDIUM_WARNING_RATE:.6f}")

        if block_reasons:
            gate_status = "BLOCK"
            gold_allowed = False
            gate_reason = "; ".join(block_reasons)
        elif warn_reasons:
            gate_status = "WARN"
            gold_allowed = True
            gate_reason = "; ".join(warn_reasons)
        else:
            gate_status = "PASS"
            gold_allowed = True
            gate_reason = "No failed rows detected above warning threshold."

        write_gate_results([{
            "run_id": batch_run_id,
            "validated_table": SILVER_TABLE,
            "run_ts_utc": batch_run_ts,
            "validation_date": validation_date,
            "validation_scope": validation_scope,
            "input_row_count": int(row_count),
            "validated_row_count": int(validated_row_count),
            "quarantined_row_count": int(quarantined_row_count),
            "quarantine_rate": float(quarantine_rate),
            "critical_failed_rows": int(critical_failed_rows),
            "high_failed_rows": int(high_failed_rows),
            "medium_failed_rows": int(medium_failed_rows),
            "low_failed_rows": int(low_failed_rows),
            "warning_rate": float(warning_rate),
            "gate_status": gate_status,
            "gate_reason": gate_reason,
            "gold_allowed": bool(gold_allowed),
            "validated_output_path": SILVER_VALIDATED_OUTPUT_PATH,
            "quarantine_output_path": SILVER_QUARANTINE_OUTPUT_PATH,
        }])

        try:
            dbutils.jobs.taskValues.set(
                key="silver_gate_status", value=gate_status)
            dbutils.jobs.taskValues.set(
                key="silver_gate_reason", value=gate_reason[:1000])
            dbutils.jobs.taskValues.set(
                key="silver_validated_path", value=SILVER_VALIDATED_OUTPUT_PATH)
            dbutils.jobs.taskValues.set(
                key="silver_quarantine_path", value=SILVER_QUARANTINE_OUTPUT_PATH)
            dbutils.jobs.taskValues.set(
                key="silver_validation_run_id", value=batch_run_id)
        except Exception:
            pass
    except Exception as exc:
        print(json.dumps({
            "layer": "silver_validate_ge",
            "message": "foreachBatch failed",
            "batch_id": int(batch_id),
            "batch_run_id": batch_run_id,
            "validation_scope": validation_scope,
            "context_root_dir": CONTEXT_ROOT_DIR,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
            "traceback": traceback.format_exc(),
        }, default=str))
        raise
    finally:
        working_df.unpersist()

In [0]:
silver_validation_query = (
    spark.readStream.table(SILVER_TABLE)
    .writeStream
    .queryName(QUERY_NAME)
    .foreachBatch(write_silver_validation_batch)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start()
)


In [0]:
for q in spark.streams.active:
    if q.name == QUERY_NAME:
        print("NAME:", q.name)
        print("ID:", q.id)
        print("IS ACTIVE:", q.isActive)
        print("STATUS:", q.status)
        print("LAST PROGRESS:", q.lastProgress)
        print("EXCEPTION:", q.exception())
        break

silver_validation_query.awaitTermination()


NAME: silver_vehicle_positions_ge_validate_stream
ID: 242019a0-24e2-451e-9305-ab450bc27203
IS ACTIVE: True
STATUS: {'message': 'Initializing StreamExecution', 'isDataAvailable': False, 'isTriggerActive': False}
LAST PROGRESS: None
EXCEPTION: None
{"layer": "silver_validate_ge", "message": "Starting foreachBatch validation", "batch_id": 7, "batch_run_id": "silver_ge_stream_7_20260422_051005", "row_count": 296909, "validation_scope": "microbatch_id=7", "context_root_dir": "/dbfs/great_expectations/silver_validate_ge_stream_ctx"}
{"layer": "silver_validate_ge", "message": "Initializing Great Expectations context", "context_root_dir": "/dbfs/great_expectations/silver_validate_ge_stream_ctx"}


/local_disk0/.ephemeral_nfs/envs/pythonEnv-96b7085d-ebae-44ae-9803-ffc6660a976a/lib/python3.12/site-packages/great_expectations/expectations/expectation.py:1519: UserWarning: `result_format` configured at the Validator-level will not be persisted. Please add the configuration to your Checkpoint config or checkpoint_run() method instead.
  warnings.warn(


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/200 [00:00<?, ?it/s]

{"layer": "silver_validate_ge", "batch_run_id": "silver_ge_stream_9_20260422_060849"}


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can